# SKU Segmentation using ABC / XYZ Analysis

In this notebook, we segment SKUs based on:
- Business importance (revenue contribution)
- Demand predictability (volatility)

This segmentation will later decide:
- Forecasting approach
- Safety stock levels
- Replenishment frequency

In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [ ]:
df = pd.read_csv("data/processed/feature_engineered_data.csv")
df["date"] = pd.to_datetime(df["date"])

## Revenue Contribution per SKU (ABC Analysis)

ABC analysis helps identify which SKUs contribute most to total revenue.

In [ ]:
df["revenue"] = df["units_sold"] * df["selling_price"]

sku_revenue = (
    df.groupby("sku_id")["revenue"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

sku_revenue["revenue_share"] = (
    sku_revenue["revenue"] / sku_revenue["revenue"].sum()
)

sku_revenue["cumulative_revenue"] = sku_revenue["revenue_share"].cumsum()

sku_revenue.head()

## Assigning ABC Categories

- A: Top ~80% of revenue
- B: Next ~15% of revenue
- C: Remaining ~5% of revenue

In [ ]:
def assign_abc(cum_rev):
    if cum_rev <= 0.80:
        return "A"
    elif cum_rev <= 0.95:
        return "B"
    else:
        return "C"

sku_revenue["ABC_class"] = sku_revenue["cumulative_revenue"].apply(assign_abc)

sku_revenue["ABC_class"].value_counts()

## Demand Variability per SKU (XYZ Analysis)

XYZ analysis classifies SKUs based on demand predictability
using the coefficient of variation (CV).

In [ ]:
sku_volatility = (
    df.groupby("sku_id")["demand_cv"]
    .mean()
    .reset_index()
)

## Assigning XYZ Categories

- X: Low variability (stable demand)
- Y: Medium variability (seasonal)
- Z: High variability (erratic demand)

In [ ]:
def assign_xyz(cv):
    if cv <= 0.5:
        return "X"
    elif cv <= 1.0:
        return "Y"
    else:
        return "Z"

sku_volatility["XYZ_class"] = sku_volatility["demand_cv"].apply(assign_xyz)

sku_volatility["XYZ_class"].value_counts()

## Combining ABC and XYZ Classifications

Each SKU is now assigned a combined segment such as:
- AX (high revenue, stable)
- AZ (high revenue, volatile)
- CZ (low revenue, unpredictable)

In [ ]:
sku_segment = (
    sku_revenue[["sku_id", "ABC_class"]]
    .merge(sku_volatility[["sku_id", "XYZ_class"]], on="sku_id", how="left")
)

sku_segment["SKU_segment"] = (
    sku_segment["ABC_class"] + sku_segment["XYZ_class"]
)

sku_segment.head()

## Distribution of SKU Segments

This helps understand how inventory complexity is distributed
across the catalogue.

In [ ]:
sku_segment["SKU_segment"].value_counts()

## Business Interpretation of SKU Segments

- **AX:** Core SKUs — high priority, tight control, frequent replenishment
- **AY:** Important but seasonal — buffer stock during peaks
- **AZ:** High value but risky — higher safety stock, careful monitoring
- **BX / BY:** Medium importance — balanced control
- **CZ:** Long tail — minimal stocking, cautious replenishment

## Attaching SKU Segments to Transactional Data

The SKU segment will be used in forecasting and inventory optimization.

In [ ]:
df_segmented = df.merge(
    sku_segment[["sku_id", "SKU_segment"]],
    on="sku_id",
    how="left"
)

df_segmented.head()

## Save Segmented Dataset

In [ ]:
df_segmented.to_csv(
    "data/processed/feature_engineered_with_segments.csv",
    index=False
)